In [ ]:
import os
from pinecone import Pinecone
from pinecone_text.sparse import BM25Encoder
from langchain_huggingface import HuggingFaceEmbeddings

def reload_rag_modules():
    import importlib
    import generator.generator as generator_module

    importlib.reload(generator_module)
    return generator_module.rag_chat, generator_module.clear_chat_history

rag_chat, clear_chat_history = reload_rag_modules()

c:\Users\thebi\Desktop\DocsGuide\src\projenv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# Load API key from environment
PINECONE_API_KEY = os.getenv("PINECONE_API_KEY")
if not PINECONE_API_KEY:
    raise ValueError("Please set the PINECONE_API_KEY environment variable!")

INDEX_NAME = "nepali-docs-hybrid"

# Initialize Pinecone index
pc = Pinecone(api_key=PINECONE_API_KEY)
index = pc.Index(INDEX_NAME)
print(f"✅ Connected to Pinecone index: {INDEX_NAME}")

# Load dense embeddings
dense_embeddings = HuggingFaceEmbeddings(model_name="universalml/Nepali_Embedding_Model")
print("✅ Dense embeddings loaded")

# Load BM25 encoder
bm25_encoder = BM25Encoder()
bm25_path = "embeddings/bm25_params.json"
if os.path.exists(bm25_path):
    bm25_encoder.load(bm25_path)
    print("✅ BM25 encoder loaded")
else:
    print(f"⚠️ BM25 parameters not found at {bm25_path}. Please generate them first!")


✅ Connected to Pinecone index: nepali-docs-hybrid
✅ Dense embeddings loaded
✅ BM25 encoder loaded


In [ ]:
# Example query
# query = "मेरो नागरिकताको प्रमाणपत्र कतै हरायो, अब त्यसको अर्को कपी (प्रतिलिपि) कसरी निकाल्न सकिन्छ?"
query="अंगीकृत नागरिकता प्राप्त गर्नका लागि नेपालमा कति समयसम्म बसेको हुनुपर्छ?"
print("\n\n" + "="*80)
print("TEST 2: BALANCED HYBRID SEARCH (alpha=0.5)")
print("="*80)

result_balanced = rag_chat(
    index=index,
    query=query,
    dense_embeddings=dense_embeddings,
    bm25_encoder=bm25_encoder,
    alpha=.5,               # Balanced hybrid (50% dense, 50% sparse)
    n_retrieval=5,
    n_generation=3
)

print("\n📝 BALANCED HYBRID ANSWER (alpha=0.5):")
print(result_balanced['answer'])
print(f"\n📚 Sources found: {len(result_balanced['sources'])}")
for i, src in enumerate(result_balanced['sources'], 1):
    print(f"  {i}. {src['source_type']}: {src['source_link']}")


# Clear history before comparison test
clear_chat_history()



TEST 2: BALANCED HYBRID SEARCH (alpha=0.5)

🔍 Original Query: अंगीकृत नागरिकता प्राप्त गर्नका लागि नेपालमा कति समयसम्म बसेको हुनुपर्छ?
🌍 Language: nepali
✅ Rewritten Query: अंगीकृत नागरिकता प्राप्त गर्नका लागि नेपालमा कति समयसम्म बसेको हुनुपर्छ?
📋 Document Type: citizenship
🏷️ Category Tag: eligibility & requirements

Generating dense vector...
Generating sparse vector (BM25)...
Performing HYBRID search (alpha=0.5) for top-5 chunks...
✅ Retrieved 5 chunks

🔥 Retrieved 5 chunks for evaluation
📝 Using top 3 chunks for answer generation

✅ Final context chunks prepared for RAG prompt:
--- Chunk 1 ---
Chunk ID: 48
Source Link: https://www.moha.gov.np/en/page/citizenship-10
Source Type: Citizenship_Faqs
Preview: Base Chunk:
प्रश्न: अंगीकृत नेपाली नागरिकता कस्तो व्यक्तिले प्राप्त गर्न सक्छन्? उत्तर: नेपाल नागरिकता ऐन, २०६३ को दफा ५ मा व्यवस्था भएबमोजिम देहायबमोजिमका व्यक्तिहरूले अंगीकृत नेपाली नागरिकताको प्रमा...

--- Chunk 2 ---
Chunk ID: 4
Source Link: https://moha.gov.np/en/post/citizen